# SPCE5025 – Spring 2026 – Exam 2  
**Initial Conditions & Given Information**  

## Epoch & Reference Frame
- **Epoch**: 01/02/2012 00:00:00 UTC  
- **Coordinate Frame**: True-of-Date (TOD)  
- **Integration Step Size**: 60 s  
- **Total Propagation**: 200 steps (12 000 s total)

## TOD Cartesian State Vector (ECI)
| Parameter | Value              | Units   |
|-----------|--------------------|---------|
| \( X \)   | 326151.080726      | m       |
| \( Y \)   | 6077471.251787     | m       |
| \( Z \)   | 2944583.918767     | m       |
| \( XD \) | -7455.178720   | m s⁻¹   |
| \( YD \) | -482.482572    | m s⁻¹   |
| \( ZD \) | 1910.883434    | m s⁻¹   |

**Altitude**: 387.046345 km

## Keplerian Elements
| Element                  | Value       | Units |
|--------------------------|-------------|-------|
| Semi-major axis \( a \)  | 6 820 000   | m     |
| Eccentricity \( e \)     | 0.010000    | –     |
| Inclination \( i \)      | 30.000000   | deg   |
| RAAN \( $\Omega$ \)        | 30.000000   | deg   |
| Argument of perigee \( $\omega$ \) | 30.000000 | deg |
| True anomaly \( $\nu$ \)   | 30.579215   | deg   |
| Period                   | 93.126394   | min   |

## Vehicle Characteristics
- Mass: 1000 kg  
- Drag area: 10 m²  
- Drag coefficient \( C_d \): 2.0  
- \( F_{10} \): 100.0 *(use **1.0** inside the Jacchia-1960 model)*

## Earth Model Parameters
- Earth rotation rate:  
  $\omega_\oplus = 72.921151467 \times 10^{-6}\ \text{rad s}^{-1}$
- Gravitational parameter:  
  $\mu_\oplus = 3.986004418 \times 10^{14}\ \text{m}^3\text{s}^{-2}$
- Equatorial radius:  
  $R_\oplus = 6\,378\,137.0\ \text{m}$
- Gravity model: **degree \( n = 2 \)**, **order \( m = 0 \)**  
- Normalized coefficients:  
  $C_{2,0} = -1.082626683553 \times 10^{-3}$
  $S_{2,0} = 0.0$

## Presentation Requirement (Problems 4–8)
Output in tabular (or CSV) form with these exact columns:

| Time (s from epoch) | $X$ (m) | $Y$ (m) | $Z$ (m) | $\dot{X}$ (m s⁻¹) | $\dot{Y}$ (m s⁻¹) | $\dot{Z}$ (m s⁻¹) | Semi-major axis $a$ (m) |
|---------------------|--------|--------|--------|------------------|------------------|------------------|-------------------------|

**Important Reminder**:  
*Integrate the initial orbit state as provided. Do not convert the vector to the perifocal frame, as that will produce incorrect results.*

**Tip for your notebook**:  

In [54]:
# Constants
from math import *
from standards import *
Pos_ = Vector3(326151.080726, 6077471.251787, 2944583.918767)
Vel_ = Vector3(-7455.178720, -482.482572, 1910.883434)

ALTD = 387.046345
Step_size = 60 #sec
Integration_duration = 12000 #sec (200 steps)
M = 1000 #kg
Drag_area = 10 #m^2
C_d = 2.0
F10 = 100 #use 1 in the jacchia 1960 model
Earth_rotation = 72.921151467e-6 #rad/sec
Earth_gravitational_parameter = 3.986004418e14 #m^3/s^2
Earth_radius = 63718137 #m
Degree = 2 #(n=2)
Order = 0 #(m=0)
C_N_M = -1.082626683553e-3
S_N_M = 2


# Problem 1

Starting with the geopotential equations from the **Class 7 notes** (slides 8–11), derive **Vallado Equation 8-51 (5th Edition)** for **n = 2** and **m = 0**. Show all work.

**Note:** Please express the vector components as **x, y, z**, rather than Vallado’s awkward unit vectors  $r_i$, $r_j$, $r_k$ 

Some general comments that may be helpful as you approach this problem:

- The fact that **m = 0** simplifies things a lot. All summations over **m** should only include the single **m = 0** term.
- Find expressions for $\sin\phi$ and $\cos\phi$ in terms of the components of the **position vector**.
- Use the gravity model coefficients provided above to state the relationship between **$J_2$** and **$\bar{C}_{20}$**.
- Expand the recursive terms for $\bar{P}_{nm}$ (verify against **Vallado Table 8-2, 5th Edition p.547**).
- Explain why the recursive relationships for $\sin(m\lambda)$, $\cos(m\lambda)$, and $m\tan\phi$ are **not needed** for this specific derivation.
- Expand the summations of the partial derivative expressions.

You have a choice of two ways to approach the problem:

1. **Evaluate the partial derivative equations** from the Class 7 notes, setting **n = 2** and **m = 0**.

2. Follow the approach suggested in **Vallado Section 8.7.1**, which evaluates the **potential function** for **n = 2** and **m = 0**, and then performs the partial derivatives with respect to $\phi$, $r$, $\lambda$

# Problem 2

Use the analytic algorithms for **Sun and Moon position** (Homework 6) to compute their positions at the following epoch:

**Tsun/moon = 01/02/2012 01:33:08 UTC**

**Note:** For simplicity, use this time directly as input to the Sun/Moon equations.  
Do **not** worry about the time offset between **UTC** and **TDB**.

In [55]:
from math import *
from standards import *

UTC_year = 2012
UTC_month = 1
UTC_day = 2
UTC_hour = 1
UTC_minute = 33
UTC_second = 8

print(f"{'Input Time:':<30} 2012-01-02T01:33:08.000Z UTC")

J_date_midnight = (
    int((1461*(UTC_year+4800+int((UTC_month-14)/12)))/4)
    +int((367*(UTC_month-2-12*int((UTC_month-14)/12)))/12)
    -int((3*((UTC_year+4900+int((UTC_month-14)/12))/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date = J_date_midnight + d
print(f"{'Julian Date:':<30} {J_date:.8f}")


## Sun calculations
#################################################################
n = J_date-2451545
L = (280.460 + 0.9856474*n)%360
g = (357.528 + 0.9856003*n)%360
Ecliptic_lon = L + 1.915*sin(radians(g))+0.020*sin(2*radians(g))
Ecliptic_lat = 0
Obliqity_of_eliptic = 23.439 - 0.0000004*n
R = 1.00014-0.01671*cos(radians(g))-0.00014*cos(2*radians(g))

x = R*cos(radians(Ecliptic_lon))
y = R*cos(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
z = R*sin(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))

m_in_au = 149597870700
sun_coordinates = Vector3(x*m_in_au, y*m_in_au, z*m_in_au)
print("Analytic Sun:")
print(sun_coordinates)
###############################################################

## Moon calculations
T = (J_date-2451545)/36525

Ecliptic_lon = 218.32+481267.881*T\
+ 6.29*sin(radians(135.0 + 477198.87*T)) - 1.27*sin(radians(259.3 - 413335.36*T))\
+ 0.66*sin(radians(235.7 + 890534.22*T)) + 0.21*sin(radians(269.9 + 954397.74*T))\
- 0.19*sin(radians(357.5 + 35999.05*T)) - 0.11*sin(radians(186.5 + 966404.03*T))

Ecliptic_lat = 5.13*sin(radians(93.3 + 483202.02*T)) + 0.28*sin(radians(228.2 + 960400.89*T))\
- 0.28*sin(radians(318.3 + 6003.15*T)) - 0.17*sin(radians(217.6 - 407332.21*T))

Pi = 0.9508 + 0.0518*cos(radians(135.0 + 477198.87*T)) + 0.0095*cos(radians(259.3 - 413335.36*T))\
+ 0.0078*cos(radians(235.7 + 890534.22*T)) + 0.0028*cos(radians(269.9 + 954397.74*T))
r = 1/sin(radians(Pi))
l = cos(radians(Ecliptic_lat))*cos(radians(Ecliptic_lon))
m = 0.9175*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) - 0.3978*sin(radians(Ecliptic_lat))
n = 0.3978*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) + 0.9175*sin(radians(Ecliptic_lat))

x = r*l
y = r*m
z = r*n

radius_earth = 6378137 #m

moon_coordinates = Vector3(x*radius_earth, y*radius_earth, z*radius_earth)
print("Analytic Moon:")
print(moon_coordinates)


Input Time:                    2012-01-02T01:33:08.000Z UTC
Julian Date:                   2455928.56467593
Analytic Sun:
Vector3(x=28168490845.921867, y=-132465810824.45862, z=-57425342373.20343)
Analytic Moon:
Vector3(x=379273674.1708138, y=113564003.996436, z=82154288.33295485)


## Problem 3

Use the **Jacchia 1960 atmosphere model** to compute the atmospheric density for the **initial satellite position vector given above**, and the **sun vector at the epoch of the initial satellite vector**.

**Note:**
- Ensure that the **Jacchia60 model** uses a value of F10 = 1.0.
- The value I specified is the value that would have come from the **NOAA solar flux report**.
- You should **divide the NOAA value by 100** for use in the **Jacchia60 atmospheric density model**.

In my sample **J60 model function**, I input a value of F10 = 100, and the function itself divides that by 100 for use in the model equations.

How you implement the code is up to you. However, the intent for the problem is that the **relevant J60 model equations use a value of F10 = 1.0.**

In [56]:
#### compute sun vector at state epoch:
UTC_year = 2012
UTC_month = 1
UTC_day = 2
UTC_hour = 0
UTC_minute = 0
UTC_second = 0

J_date_midnight = (
    int((1461*(UTC_year+4800+int((UTC_month-14)/12)))/4)
    +int((367*(UTC_month-2-12*int((UTC_month-14)/12)))/12)
    -int((3*((UTC_year+4900+int((UTC_month-14)/12))/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date = J_date_midnight + d

#### Sun calculations
n = J_date-2451545
L = (280.460 + 0.9856474*n)%360
g = (357.528 + 0.9856003*n)%360
Ecliptic_lon = L + 1.915*sin(radians(g))+0.020*sin(2*radians(g))
Ecliptic_lat = 0
Obliqity_of_eliptic = 23.439 - 0.0000004*n
R = 1.00014-0.01671*cos(radians(g))-0.00014*cos(2*radians(g))

x = R*cos(radians(Ecliptic_lon))
y = R*cos(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
z = R*sin(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))

m_in_au = 149597870700
sun_coordinates = Vector3(x*m_in_au, y*m_in_au, z*m_in_au)

#### compute atmospheric drag
w_ = Vector3(0, 0, 72.921151467e-6)
v_r_ = Vel_ - (w_.cross(Pos_))
B = 0.55 #radians
S_ = sun_coordinates.unit_vector()
U_ = Vector3(S_.x*cos(B)-S_.y*sin(B), S_.y*cos(B)+S_.x*sin(B), S_.z)
h = ALTD/1.852

cos_angle_from_sv_to_diurnal_bulge = Pos_.dot(U_) / (Pos_.magnitude()*U_.magnitude())
print(f"Angle from SV to bulge:\n{degrees(acos((cos_angle_from_sv_to_diurnal_bulge))):.6}")
F10_scaled = F10/100
p_0 = exp((6.363*exp(-0.0048*h)-0.00368*h-15.738)*log(10))
p = p_0*(0.85*F10_scaled)*(1+0.02375*(exp(0.0102*h)-1.9)*(1+cos_angle_from_sv_to_diurnal_bulge)**3)*515.37886
print(f"Atmospheric Density:\n{p:.6}")


Angle from SV to bulge:
137.759
Atmospheric Density:
2.9452e-12
